# Bangla Multimodal Fake News: Preprocessing

This notebook prepares the BTMD dataset. It does **not** train any model, so a CPU runtime is enough.

Steps: mount Drive, unpack the code, check the code version, run preflight, build the unified table, make the group split, and run sanity checks.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. Unpack the project code

The unpack cell always removes the old copy first, so stale code cannot run.

In [2]:
import shutil
import zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
CODE_DIR = Path("/content/Bangla-Multimodal-FakeNews")

candidates = list(DRIVE_ROOT.glob("Bangla-Multimodal-FakeNews*.zip")) + list(
    (DRIVE_ROOT / "BFND-data").glob("Bangla-Multimodal-FakeNews*.zip")
)
assert candidates, "Code zip not found on Drive. Upload Bangla-Multimodal-FakeNews.zip first."
ZIP = max(candidates, key=lambda p: p.stat().st_mtime)
print("Using zip:", ZIP)

shutil.rmtree(CODE_DIR, ignore_errors=True)   # remove any old copy so stale code cannot run
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
print("Files in zip:", len(zipfile.ZipFile(ZIP).namelist()))

Using zip: /content/drive/MyDrive/Bangla-Multimodal-FakeNews.zip
Files in zip: 74


In [3]:
import os
from pathlib import Path

os.chdir(CODE_DIR)
config_text = Path("configs/config.py").read_text(encoding="utf-8")
assert "CLIP_IMAGE_MODEL" in config_text, (
    "Old code detected. Upload the latest zip to Drive, then run the unpack cell again."
)
print("Code version OK. Working directory:", os.getcwd())

Code version OK. Working directory: /content/Bangla-Multimodal-FakeNews


## 2. Install packages

Torch is already installed in Colab, so it is not installed here.

In [4]:
!pip install -q "pandas>=2.0" "scikit-learn>=1.3" "pyarrow>=14.0" "openpyxl>=3.1" "rank-bm25>=0.2.2" joblib pytest

## 3. Run the unit tests (no data needed)

In [5]:
!python -m pytest -q tests/

........................................................................ [ 78%]
....................                                                     [100%]
92 passed in 19.51s


## 4. Preflight

Checks model IDs, schema, data files and the GPU. A GPU WARN is expected in this notebook. Any FAIL must be fixed before continuing.

In [6]:
!python scripts/00_preflight.py

[OK]   Hugging Face model: csebuetnlp/banglabert
[OK]   Hugging Face model: sentence-transformers/clip-ViT-B-32
[OK]   Hugging Face model: sentence-transformers/clip-ViT-B-32-multilingual-v1
[OK]   Hugging Face model: intfloat/multilingual-e5-base
[OK]   Hugging Face model: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
[WARN] btmd_schema.py: 'source_url_col' is not set; related features/guards will be skipped
[OK]   btmd_schema.py: required fields are filled in
[OK]   BTMD slim data: /content/drive/MyDrive/BFND-data/data/raw
[OK] evidence.csv: /content/drive/MyDrive/BFND-data/data/evidence/evidence.csv
[MISS] cultural_annotations.csv (manual): /content/drive/MyDrive/BFND-data/data/annotations/cultural_annotations.csv
[OK] cultural_silver_labels.csv (LLM): /content/drive/MyDrive/BFND-data/data/annotations/cultural_silver_labels.csv
[MISS] claim_evidence_pairs.csv: /content/drive/MyDrive/BFND-data/data/annotations/claim_evidence_pairs.csv
[OK]   GPU: Tesla T4

Experiments tha

## 5. Build the unified table and the group split

In [7]:
!python scripts/01_prepare_data.py

label  Authentic  Misleading  Satire
split                               
test         337         219      32
train       1531        1074     131
val          340         216      30

Data stats: {'rows_raw': 6058, 'excluded_by_label': {'Clickbait': 576}, 'group_source': "column 'heading' (text hash when empty)", 'image_value_present_but_file_missing': 0, 'dropped_empty_text': 1566, 'dropped_duplicate_text_label': 1, 'dropped_duplicate_image': 5, 'rows_final': 3910, 'rows_with_image': 2397, 'label_counts': {'Authentic': 2208, 'Misleading': 1509, 'Satire': 193}}


## 6. Data statistics

In [8]:
import json
import pandas as pd
from configs.config import PROCESSED_DIR, RAW_DIR

stats = json.loads((PROCESSED_DIR / "data_stats.json").read_text(encoding="utf-8"))
for key, value in stats.items():
    print(f"{key}: {value}")

rows_raw: 6058
excluded_by_label: {'Clickbait': 576}
group_source: column 'heading' (text hash when empty)
image_value_present_but_file_missing: 0
dropped_empty_text: 1566
dropped_duplicate_text_label: 1
dropped_duplicate_image: 5
rows_final: 3910
rows_with_image: 2397
label_counts: {'Authentic': 2208, 'Misleading': 1509, 'Satire': 193}


## 7. Label mapping check

Each raw label should map to the expected project label.

In [9]:
unified = pd.read_csv(PROCESSED_DIR / "unified.csv")
print(pd.crosstab(unified["label_raw"], unified["label"], margins=True))

label           Authentic  Misleading  Satire   All
label_raw                                          
Misinformation          0        1509       0  1509
Real                 2208           0       0  2208
Satire                  0           0     193   193
All                  2208        1509     193  3910


## 8. Split check

Every split should contain all three classes.

In [10]:
split = pd.read_csv(PROCESSED_DIR / "unified_split.csv")
print(pd.crosstab(split["split"], split["label"], margins=True))

label  Authentic  Misleading  Satire   All
split                                     
test         337         219      32   588
train       1531        1074     131  2736
val          340         216      30   586
All         2208        1509     193  3910


## 9. Manual sample check

Read the 20 texts and check that each label looks right. Note any broken or suspicious rows.

In [11]:
pd.set_option("display.max_colwidth", 120)
print(unified.sample(20, random_state=7)[["sample_id", "label_raw", "label", "has_image", "text"]].to_string())

                                 sample_id       label_raw       label  has_image                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

## 10. Confound checks

Compare text length and source across classes. If they differ a lot, a model may learn the shortcut instead of the content.

In [12]:
unified["n_chars"] = unified["text"].str.len()
print("Text length (characters) by class:")
print(unified.groupby("label")["n_chars"].describe())

raw = pd.read_parquet(RAW_DIR / "data" / "metadata.parquet")
print("\nSource category by raw label:")
print(pd.crosstab(raw["multi_category"], raw["source_category"], margins=True))

Text length (characters) by class:
             count         mean          std   min   25%     50%     75%  \
label                                                                      
Authentic   2208.0  1428.651268  1563.552462   9.0  79.0  1136.0  2145.5   
Misleading  1509.0   438.131875   834.043531  11.0  64.0   100.0   342.0   
Satire       193.0   213.321244   532.602015   2.0  40.0    68.0   123.0   

                max  
label                
Authentic   17840.0  
Misleading   8616.0  
Satire       3934.0  

Source category by raw label:
source_category  Blog  Facebook  Instagram  Miscellaneous  News Channel  \
multi_category                                                            
Clickbait           0       100          0             69             3   
Misinformation      2      1383         10             27            34   
Real                0      1558          1              1           237   
Satire              0       586          3              0           

Next: open `02_model_training.ipynb` with a GPU runtime.